<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_transformer_reID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# 演示：如何使用我们的姿态 Transformer 进行动物的无监督身份跟踪
![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1628250004229-KVYD7JJVHYEFDJ32L9VJ/DLClogo2021.jpg?format=1000w)

https://github.com/DeepLabCut/DeepLabCut

### 本 Notebook 演示了如何为多动物 DeepLabCut (maDLC) 演示项目（三只小鼠）使用 Transformer：
- 加载包含预训练模型和未标记视频的 mini-demo 数据。
- 分析新的视频。
- 使用 Transformer 进行无监督的身份（ID）跟踪。
- 创建质量检查图和视频。

### 如需创建完整的 maDLC 流程，请参阅我们的完整文档：https://deeplabcut.github.io/DeepLabCut/README.html
- 特别值得关注的是关于 maDLC 的完整操作指南：https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html
- maDLC 快速指南：https://deeplabcut.github.io/DeepLabCut/docs/tutorial.html
- 一个演示 COLAB 笔记本，介绍如何在您自己的数据上使用 maDLC：https://github.com/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_maDLC_TrainNetwork_VideoAnalysis.ipynb

### 要开始使用，请前往 "Runtime" -> "change runtime type" -> 选择 "Python3"，然后选择 "GPU"

‼️ **注意：此演示适用于 maDLC 2.2 版本**

In [ ]:
# Install DLC version 2.2-2.3 (pre DLC3):
!pip install "deeplabcut[tf]"

In [3]:
import deeplabcut
import os

## 重要提示 - 请重启运行时环境以导入已更新的包！

请在继续操作之前，**点击上方输出中的“重启运行时”**！

下面的单元格中**无需编辑任何信息**，您只需依次点击运行即可：

### 从我们的服务器下载演示项目：

In [5]:
# Download our demo project:
import requests
from io import BytesIO
from zipfile import ZipFile

url_record = "https://zenodo.org/api/records/7883589"
response = requests.get(url_record)
if response.status_code == 200:
    file = response.json()["files"][0]
    title = file["key"]
    print(f"Downloading {title}...")
    with requests.get(file["links"]["self"], stream=True) as r:
        with ZipFile(BytesIO(r.content)) as zf:
            zf.extractall(path="/content")
else:
    raise ValueError(f"The URL {url_record} could not be reached.")

## 使用我们的 maDLC DLCRNet 分析 3 只小鼠视频（该网络预训练于 3 只小鼠的数据集）

由于设置了 `auto_track=True`，您只需一步操作，即可完成检测、关联成本计算、创建轨迹片段（tracklets）以及将它们拼接起来。我们可以使用这些结果与下方的 Transformer 引导式跟踪方法进行比较。

In [6]:
project_path = "/content/demo-me-2021-07-14"
config_path = os.path.join(project_path, "config.yaml")
video = os.path.join(project_path, "videos", "videocompressed1.mp4")

In [7]:
deeplabcut.analyze_videos(config_path,[video],
                          shuffle=0, videotype="mp4",
                          auto_track=True)

Using snapshot-20000 for model /content/demo-me-2021-07-14/dlc-models/iteration-0/demoJul14-trainset95shuffle0


/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Activating extracting of PAFs
Starting to analyze %  /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Loading  /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Duration of video [s]:  77.67 , recorded with  30.0 fps!
Overall # of frames:  2330  found with (before cropping) frame dimensions:  640 480
Starting to extract posture from the video(s) with batchsize: 8


100%|██████████| 2330/2330 [00:39<00:00, 58.83it/s]


Video Analyzed. Saving results in /content/demo-me-2021-07-14/videos...


...

100%|██████████| 2330/2330 [00:02<00:00, 1088.72it/s]
2330it [00:06, 342.29it/s] 


The tracklets were created (i.e., under the hood deeplabcut.convert_detections2tracklets was run). Now you can 'refine_tracklets' in the GUI, or run 'deeplabcut.stitch_tracklets'.
Processing...  /content/demo-me-2021-07-14/videos/videocompressed1.mp4


100%|██████████| 4/4 [00:00<00:00, 1488.53it/s]
/usr/local/lib/python3.11/dist-packages/deeplabcut/refine_training_dataset/stitch.py:934: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  df.to_hdf(output_name, "tracks", format="table", mode="w")


The videos are analyzed. Time to assemble animals and track 'em... 
 Call 'create_video_with_all_detections' to check multi-animal detection quality before tracking.
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.


'DLC_dlcrnetms5_demoJul14shuffle0_20000'

### 接下来，你将逐帧计算局部的、时空分组，并追踪身体部位的组合体：

## 创建漂亮的视频输出：

In [8]:
#Filter the predictions to remove small jitter, if desired:
deeplabcut.filterpredictions(config_path, [video], shuffle=0, videotype="mp4")
deeplabcut.create_labeled_video(
    config_path,
    [video],
    videotype="mp4",
    shuffle=0,
    color_by="individual",
    keypoints_only=False,
    draw_skeleton=True,
    filtered=True,
)

Filtering with median model /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Saving filtered csv poses!


/usr/local/lib/python3.11/dist-packages/deeplabcut/post_processing/filtering.py:298: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  data.to_hdf(outdataname, "df_with_missing", format="table", mode="w")


Starting to process video: /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Loading /content/demo-me-2021-07-14/videos/videocompressed1.mp4 and data.
Duration of video [s]: 77.67, recorded with 30.0 fps!
Overall # of frames: 2330 with cropped frame dimensions: 640 480
Generating frames and creating video.


/usr/local/lib/python3.11/dist-packages/deeplabcut/utils/make_labeled_video.py:140: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Dataframe.groupby(level="individuals", axis=1).size().values // 3
100%|██████████| 2330/2330 [00:31<00:00, 73.04it/s]


[True]

现在，在左侧面板中，如果您点击文件夹图标，您将看到项目文件夹 `"demo-me.."`；点击进入该文件夹，然后进入 `"videos"` 目录，您会找到 `"..._id_labeled.mp4"` 视频文件。您可以双击该文件以下载并进行检查！

### 创建数据的图表：

> 运行结束后，你可以在 `videos` 和 `plot-poses` 文件夹中查看轨迹！ (有时你需要点击文件夹的刷新图标才能看到它们)。在这些文件夹中，例如，查看 `plotmus1.png` 可以看到身体部件随时间变化的像素位置图。

In [9]:
deeplabcut.plot_trajectories(config_path, [video], shuffle=0,videotype="mp4")

Loading  /content/demo-me-2021-07-14/videos/videocompressed1.mp4 and data.
Plots created! Please check the directory "plot-poses" within the video directory


# reID 的 Transformer

尽管这里的跟踪效果在不使用 Transformer 的情况下已经非常好，但我们希望向您演示一下工作流程！

In [10]:
deeplabcut.transformer_reID(
    config_path,
    [video],
    shuffle=0,
    videotype="mp4",
    track_method="ellipse",
    n_triplets=100,
)

Using snapshot-20000 for model /content/demo-me-2021-07-14/dlc-models/iteration-0/demoJul14-trainset95shuffle0


/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '
/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use 

Activating extracting of PAFs
Starting to analyze %  /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Loading  /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Duration of video [s]:  77.67 , recorded with  30.0 fps!
Overall # of frames:  2330  found with (before cropping) frame dimensions:  640 480
Starting to extract posture


100%|██████████| 2330/2330 [01:18<00:00, 29.78it/s]


If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.
Epoch 10, train acc: 0.61
Epoch 10, test acc 0.45
Epoch 20, train acc: 0.74
Epoch 20, test acc 0.65
Epoch 30, train acc: 0.78
Epoch 30, test acc 0.55
Epoch 40, train acc: 0.76
Epoch 40, test acc 0.50
Epoch 50, train acc: 0.85
Epoch 50, test acc 0.55
Epoch 60, train acc: 0.84
Epoch 60, test acc 0.60
Epoch 70, train acc: 0.85
Epoch 70, test acc 0.55
Epoch 80, train acc: 0.79
Epoch 80, test acc 0.55
Epoch 90, train acc: 0.88
Epoch 90, test acc 0.55
Epoch 100, train acc: 0.84
Epoch 100, test acc 0.55
loading params
Processing...  /content/demo-me-2021-07-14/videos/videocompressed1.mp4


100%|██████████| 4/4 [00:00<00:00, 483.21it/s]
/usr/local/lib/python3.11/dist-packages/deeplabcut/refine_training_dataset/stitch.py:934: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  df.to_hdf(output_name, "tracks", format="table", mode="w")


现在我们可以使用**Transformer 引导式跟踪**（transformer-guided tracking）来制作另一个视频。

In [11]:
deeplabcut.plot_trajectories(
    config_path,
    [video],
    shuffle=0,
    videotype="mp4",
    track_method="transformer",
)

Loading  /content/demo-me-2021-07-14/videos/videocompressed1.mp4 and data.
Plots created! Please check the directory "plot-poses" within the video directory


In [12]:
deeplabcut.create_labeled_video(
    config_path,
    [video],
    videotype="mp4",
    shuffle=0,
    color_by="individual",
    keypoints_only=False,
    draw_skeleton=True,
    track_method="transformer"
)

Starting to process video: /content/demo-me-2021-07-14/videos/videocompressed1.mp4
Loading /content/demo-me-2021-07-14/videos/videocompressed1.mp4 and data.
Duration of video [s]: 77.67, recorded with 30.0 fps!
Overall # of frames: 2330 with cropped frame dimensions: 640 480
Generating frames and creating video.


/usr/local/lib/python3.11/dist-packages/deeplabcut/utils/make_labeled_video.py:140: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  Dataframe.groupby(level="individuals", axis=1).size().values // 3
100%|██████████| 2330/2330 [00:31<00:00, 73.75it/s]


[True]